In [1]:
# Imports
import sys
import logging
from datetime import datetime
import pandas as pd
from IPython.display import display

sys.path.insert(0, '../../../LOGOS')
from src import Pert, plot_gantt_chart, plot_resource_utilization, plot_location_utilization, plot_equipment_utilization
# Configure logging in the runner (avoid setting basicConfig inside the module)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
import json
from pathlib import Path

cwd = Path.cwd()
benchmark = cwd/'benchmarks'
benchmark_results_file = benchmark/'priority_rules_results.json'

## Load Benchmark results
with open(benchmark_results_file, "r", encoding="utf-8") as f:
    benchmark_data = json.load(f)

In [2]:
def run_case(case_name, file_name, json_path,
             schema_file="outage_schema.json",
             benchmark_data=None):
    """
    Runs scheduling comparison for a given case and file.

    Parameters:
        case_name (str): Case identifier (e.g., 'j60')
        file_name (str): File name (e.g., 'j601_1.sm')
        json_path (str): Path to JSON file for Pert model
        schema_file (str): Path to schema file (default: outage_schema.json)
        benchmark_data (dict): Benchmark dataset for RCPSP comparison

    Returns:
        results_df (pd.DataFrame): LOGOS.CPM results
        data_df (pd.DataFrame): RCPSP benchmark results
    """

    results_sgs = {}
    results_pgs = {}
    results_pgs_pr = {}


    # Load Pert Model
    pert = Pert.from_json_file(json_path, schema_path=schema_file)

    prs = [
        'es','ef','ls','lf', 'duration','random',
        'mts', 'mtp', 'grpw', 'grd', 'rr', 'avgrr',
        'maxrr', 'minrr','mehh_8000_b','mehh_3375_b',
        'mehh_1000_b','mehh_125_b','gphh_b'
    ]

    # Compute results with Serial
    for rule in prs:
        out = pert.calculateSerialScheduleWithResources(priority_rule=rule)
        results_sgs[rule] = out['scheduled_duration'] - 2  # remove start/end duration
        violations, is_feasible = pert.check_dependency_violations()
        if not is_feasible:
            print(violations)
    sgs = ['first', 'max_use_res_ranked', 'max_use_res_shuffled', 'md_knapsack', 'look_ahead']

    # Compute results with Parallel
    for s in sgs:
        out = pert.calculateScheduleWithResources(sgs=s)
        results_pgs[s] = out['scheduled_duration'] - 2  # remove start/end duration
        violations, is_feasible = pert.check_dependency_violations()
        if not is_feasible:
            print(violations)

    for s in sgs:
        results_pgs_pr[s] = {}
        for rule in prs:
            out = pert.calculateScheduleWithResources(sgs=s, priority_rule=rule)
            results_pgs_pr[s][rule] = out['scheduled_duration'] - 2  # remove start/end duration
            violations, is_feasible = pert.check_dependency_violations()
            if not is_feasible:
                print(violations)



    print('Results from LOGOS.CPM Using Serial Generation Scheme:')
    print('-' * 60)
    results_df_sgs = pd.DataFrame(results_sgs, index=[0])
    display(results_df_sgs)

    if benchmark_data is None:
        raise ValueError("benchmark_data must be provided")

    data = benchmark_data[case_name][file_name]
    data_df = pd.DataFrame(data, index=[0]).filter(like='serial_forward')
    data_df.columns = data_df.columns.str.replace("_serial_forward", "", regex=False)
    data_df.columns = data_df.columns.str.lower()

    display(data_df)

    print('Results from LOGOS.CPM Using Parallel Generation Scheme:')
    print('-' * 60)
    results_df_pgs = pd.DataFrame(results_pgs, index=[0])
    display(results_df_pgs)

    for s in sgs:
        print(f'Results from LOGOS.CPM Using Parallel Generation Scheme "{s}" with Priority Rule:')
        print('-' * 60)
        results_df_pgs_pr = pd.DataFrame(results_pgs_pr[s], index=[0])
        display(results_df_pgs_pr)

    # RCPSP benchmark comparison
    print('Results from RCPSP')
    print('-' * 60)

    data_df = pd.DataFrame(data, index=[0]).filter(like='parallel_forward')
    data_df.columns = data_df.columns.str.replace("_parallel_forward", "", regex=False)
    data_df.columns = data_df.columns.str.lower()

    display(data_df)


    return results_df_sgs, results_df_pgs, data_df

## Scheduling with 30 activities

In [3]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j30',
    file_name='j301_1.sm',
    json_path='j301_1.json',
    benchmark_data=benchmark_data
)

INFO:src.CPM.validate_outage_data:OUTAGE DATA VALIDATION
INFO:src.CPM.validate_outage_data:✓ Schema validation passed
INFO:src.CPM.validate_outage_data:✓ All task IDs are unique
INFO:src.CPM.validate_outage_data:✓ All resource skill IDs are unique
INFO:src.CPM.validate_outage_data:✓ All equipment IDs are unique
INFO:src.CPM.validate_outage_data:✓ All location IDs are unique
INFO:src.CPM.validate_outage_data:✓ Task references checked
INFO:src.CPM.validate_outage_data:✓ Location references are valid
INFO:src.CPM.validate_outage_data:✓ Equipment references are valid
INFO:src.CPM.validate_outage_data:✓ Skill type references are valid
INFO:src.CPM.validate_outage_data:✓ Hold-point logic is valid
INFO:src.CPM.validate_outage_data:✓ No circular dependencies found
INFO:src.CPM.validate_outage_data:✓ VALIDATION PASSED
INFO:src.CPM.pert:Starting Serial SGS | activities=32 | CPM=40.0h | rule=es
INFO:src.CPM.pert:Serial SGS complete | CPM=40.0h | actual=55.0h | delay=71.0h | scheduled=32/32 | rule

Results from LOGOS.CPM Using Serial Generation Scheme:
------------------------------------------------------------


,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,53.0,60.0,46.0,49.0,57.0,47.0,49.0,74.0,63.0,53.0,73.0,55.0,55.0,73.0,52.0,53.0,52.0,46.0,74.0


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,51,60,46,49,57,49,49,61,60,53,52,53,52,46,74


Results from LOGOS.CPM Using Parallel Generation Scheme:
------------------------------------------------------------


...

,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0


Results from LOGOS.CPM Using Parallel Generation Scheme "look_ahead" with Priority Rule:
------------------------------------------------------------


,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,51.0,51.0,46.0,43.0,45.0,53.0,43.0,61.0,61.0,46.0,61.0,46.0,46.0,61.0,46.0,51.0,46.0,43.0,61.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,irsm,acs,wcs,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,61,51,46,43,45,61,43,53,61,53,45,45,43,46,51,46,43,61



## Scheduling with 60 activities

In [4]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j60',
    file_name='j601_1.sm',
    json_path='j601_1.json',
    benchmark_data=benchmark_data
)

INFO:src.CPM.validate_outage_data:OUTAGE DATA VALIDATION
INFO:src.CPM.validate_outage_data:✓ Schema validation passed
INFO:src.CPM.validate_outage_data:✓ All task IDs are unique
INFO:src.CPM.validate_outage_data:✓ All resource skill IDs are unique
INFO:src.CPM.validate_outage_data:✓ All equipment IDs are unique
INFO:src.CPM.validate_outage_data:✓ All location IDs are unique
INFO:src.CPM.validate_outage_data:✓ Task references checked
INFO:src.CPM.validate_outage_data:✓ Location references are valid
INFO:src.CPM.validate_outage_data:✓ Equipment references are valid
INFO:src.CPM.validate_outage_data:✓ Skill type references are valid
INFO:src.CPM.validate_outage_data:✓ Hold-point logic is valid
INFO:src.CPM.validate_outage_data:✓ No circular dependencies found
INFO:src.CPM.validate_outage_data:✓ VALIDATION PASSED
INFO:src.CPM.pert:Starting Serial SGS | activities=62 | CPM=79.0h | rule=es
INFO:src.CPM.pert:Serial SGS complete | CPM=79.0h | actual=88.0h | delay=158.0h | scheduled=62/62 | rul

Results from LOGOS.CPM Using Serial Generation Scheme:
------------------------------------------------------------


,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,86.0,88.0,77.0,77.0,121.0,98.0,77.0,102.0,88.0,98.0,102.0,100.0,100.0,102.0,77.0,85.0,77.0,109.0,121.0


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,86,88,77,77,121,80,77,106,84,98,77,85,77,109,121


Results from LOGOS.CPM Using Parallel Generation Scheme:
------------------------------------------------------------


...

,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0


Results from LOGOS.CPM Using Parallel Generation Scheme "look_ahead" with Priority Rule:
------------------------------------------------------------


,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,82.0,85.0,86.0,86.0,85.0,82.0,82.0,87.0,92.0,90.0,85.0,86.0,86.0,85.0,86.0,85.0,86.0,87.0,85.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,irsm,acs,wcs,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,85,85,86,86,85,84,82,85,86,92,85,85,85,86,85,86,96,96


## Scheduling with 90 activities

In [5]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j90',
    file_name='j901_1.sm',
    json_path='j901_1.json',
    benchmark_data=benchmark_data
)

INFO:src.CPM.validate_outage_data:OUTAGE DATA VALIDATION
INFO:src.CPM.validate_outage_data:✓ Schema validation passed
INFO:src.CPM.validate_outage_data:✓ All task IDs are unique
INFO:src.CPM.validate_outage_data:✓ All resource skill IDs are unique
INFO:src.CPM.validate_outage_data:✓ All equipment IDs are unique
INFO:src.CPM.validate_outage_data:✓ All location IDs are unique
INFO:src.CPM.validate_outage_data:✓ Task references checked
INFO:src.CPM.validate_outage_data:✓ Location references are valid
INFO:src.CPM.validate_outage_data:✓ Equipment references are valid
INFO:src.CPM.validate_outage_data:✓ Skill type references are valid
INFO:src.CPM.validate_outage_data:✓ Hold-point logic is valid
INFO:src.CPM.validate_outage_data:✓ No circular dependencies found
INFO:src.CPM.validate_outage_data:✓ VALIDATION PASSED
INFO:src.CPM.pert:Starting Serial SGS | activities=92 | CPM=69.0h | rule=es
INFO:src.CPM.pert:Serial SGS complete | CPM=69.0h | actual=90.0h | delay=261.0h | scheduled=92/92 | rul

Results from LOGOS.CPM Using Serial Generation Scheme:
------------------------------------------------------------


,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,88.0,98.0,83.0,82.0,111.0,105.0,91.0,143.0,92.0,111.0,148.0,97.0,97.0,148.0,84.0,101.0,84.0,102.0,148.0


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,94,98,83,82,111,88,89,101,95,108,84,101,84,102,148


Results from LOGOS.CPM Using Parallel Generation Scheme:
------------------------------------------------------------


...

,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0,97.0


Results from LOGOS.CPM Using Parallel Generation Scheme "look_ahead" with Priority Rule:
------------------------------------------------------------


,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,89.0,87.0,81.0,81.0,95.0,86.0,86.0,97.0,92.0,87.0,90.0,89.0,89.0,90.0,81.0,87.0,81.0,87.0,90.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,irsm,acs,wcs,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,85,90,81,81,97,94,86,92,89,93,89,91,83,81,91,81,89,97


## Scheduling with 120 activities

In [6]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j120',
    file_name='j1201_1.sm',
    json_path='j1201_1.json',
    benchmark_data=benchmark_data
)

INFO:src.CPM.validate_outage_data:OUTAGE DATA VALIDATION
INFO:src.CPM.validate_outage_data:✓ Schema validation passed
INFO:src.CPM.validate_outage_data:✓ All task IDs are unique
INFO:src.CPM.validate_outage_data:✓ All resource skill IDs are unique
INFO:src.CPM.validate_outage_data:✓ All equipment IDs are unique
INFO:src.CPM.validate_outage_data:✓ All location IDs are unique
INFO:src.CPM.validate_outage_data:✓ Task references checked
INFO:src.CPM.validate_outage_data:✓ Location references are valid
INFO:src.CPM.validate_outage_data:✓ Equipment references are valid
INFO:src.CPM.validate_outage_data:✓ Skill type references are valid
INFO:src.CPM.validate_outage_data:✓ Hold-point logic is valid
INFO:src.CPM.validate_outage_data:✓ No circular dependencies found
INFO:src.CPM.validate_outage_data:✓ VALIDATION PASSED
INFO:src.CPM.pert:Starting Serial SGS | activities=122 | CPM=101.0h | rule=es
INFO:src.CPM.pert:Serial SGS complete | CPM=101.0h | actual=139.0h | delay=613.0h | scheduled=122/122

Results from LOGOS.CPM Using Serial Generation Scheme:
------------------------------------------------------------


,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,137.0,143.0,119.0,123.0,136.0,165.0,126.0,178.0,148.0,154.0,195.0,148.0,148.0,195.0,124.0,138.0,114.0,157.0,193.0


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,132,144,119,123,147,123,125,153,140,156,124,138,114,157,193


Results from LOGOS.CPM Using Parallel Generation Scheme:
------------------------------------------------------------


...

,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0,148.0


Results from LOGOS.CPM Using Parallel Generation Scheme "look_ahead" with Priority Rule:
------------------------------------------------------------


,es,ef,ls,lf,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,131.0,130.0,120.0,126.0,124.0,120.0,121.0,150.0,177.0,133.0,181.0,133.0,133.0,181.0,124.0,128.0,120.0,142.0,178.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,irsm,acs,wcs,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,127,128,120,126,132,121,121,136,134,133,126,125,123,124,130,120,142,178
